# This is where we can train our own model

In [111]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

Load the data for training

In [112]:
df = pd.read_csv("../data/train_images.csv")
attributes = np.load("../data/attributes.npy", allow_pickle=True)
attributes = (attributes - attributes.min(axis=1, keepdims=True)) / (
    attributes.max(axis=1, keepdims=True) - attributes.min(axis=1, keepdims=True)
)

In [113]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Replace with your data
class_names = [i + 1 for i in range(200)]

scaler = StandardScaler()
attrs_scaled = scaler.fit_transform(attributes)

pca = PCA(n_components=65)
pca_2d = pca.fit_transform(attrs_scaled)

print(f'Cumulative variance: {np.sum(pca.explained_variance_ratio_):.1%}')

attributes = pca_2d

Cumulative variance: 91.9%


In [114]:
class ImageAttributeDataset(Dataset):
    def __init__(self, data_df, attrs, transform):
        self.data = data_df
        self.transform = transform
        self.attributes = attrs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = "../data" + self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label']
        # subtract 1 from label to go to attribute index
        attrs = torch.tensor(self.attributes[label - 1], dtype=torch.float32)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, attrs


In [115]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [116]:
print(train_df)

                  image_path  label
1249  /train_images/1250.jpg     42
3882  /train_images/3883.jpg    193
686    /train_images/687.jpg     23
1452  /train_images/1453.jpg     49
2357  /train_images/2358.jpg     85
...                      ...    ...
3730  /train_images/3731.jpg    173
1353  /train_images/1354.jpg     46
1938  /train_images/1939.jpg     68
444    /train_images/445.jpg     15
1453  /train_images/1454.jpg     49

[3140 rows x 2 columns]


In [119]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = ImageAttributeDataset(train_df, transform=train_tfms, attrs=attributes)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [120]:
classes = train_df['label'].unique()

Train the model

In [121]:
class ImageEmbeddingModel(nn.Module):
    def __init__(self, embed_dim=65, img_size=224):  # Smaller input!
        super().__init__()
        self.backbone = nn.Sequential(
            # Standard 3x3 kernels, stride=1 → MUCH faster
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64x64
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 32x32
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 16x16
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)  # 8x8
        )

        # Precompute flat size for 128x128 input
        with torch.no_grad():
            x = torch.randn(1, 3, img_size, img_size)
            x = self.backbone(x)
            flat_size = x.numel() // x.shape[0]  # ~16k vs your 50k

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 512),  # Smaller hidden layer
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, embed_dim)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


In [122]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImageEmbeddingModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.MSELoss()
# Training
model.train()
for epoch in range(40):  # Adjust epochs
    total_loss = 0
    for images, attrs in train_loader:

        images, attrs = images.to(device), attrs.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, attrs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}')

print("Finished training")


Epoch 1, Loss: 4.7163
Epoch 2, Loss: 4.7088
Epoch 3, Loss: 4.6683
Epoch 4, Loss: 4.6502
Epoch 5, Loss: 4.6246
Epoch 6, Loss: 4.6076
Epoch 7, Loss: 4.5496
Epoch 8, Loss: 4.5051
Epoch 9, Loss: 4.4448
Epoch 10, Loss: 4.3215
Epoch 11, Loss: 4.2476
Epoch 12, Loss: 4.1047
Epoch 13, Loss: 3.9939
Epoch 14, Loss: 3.8038
Epoch 15, Loss: 3.7172
Epoch 16, Loss: 3.6364
Epoch 17, Loss: 3.4623
Epoch 18, Loss: 3.3041
Epoch 19, Loss: 3.2462
Epoch 20, Loss: 3.1255
Epoch 21, Loss: 3.0633
Epoch 22, Loss: 2.9612
Epoch 23, Loss: 2.8607
Epoch 24, Loss: 2.7941
Epoch 25, Loss: 2.7549
Epoch 26, Loss: 2.6649
Epoch 27, Loss: 2.5980
Epoch 28, Loss: 2.5560
Epoch 29, Loss: 2.5284
Epoch 30, Loss: 2.4647
Epoch 31, Loss: 2.4303
Epoch 32, Loss: 2.3780
Epoch 33, Loss: 2.3363
Epoch 34, Loss: 2.3535
Epoch 35, Loss: 2.2603
Epoch 36, Loss: 2.2581
Epoch 37, Loss: 2.2305
Epoch 38, Loss: 2.1916
Epoch 39, Loss: 2.1556
Epoch 40, Loss: 2.1036
Finished training


Run the model on the val dataset

In [123]:
print(len(val_df))

786


In [124]:
val_dataset = ImageAttributeDataset(val_df, transform=test_tfms, attrs=attributes)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

val_loss = 0
criterion = nn.CosineEmbeddingLoss()
predictions = []
with torch.no_grad():
    for images, attrs in val_loader:
        images, attrs = images.to(device), attrs.to(device)
        preds = model(images)
        predictions.extend(preds.tolist())
        target = torch.ones(preds.size(0)).to(device)
        loss = criterion(preds, attrs, target)
        val_loss += loss.item()
val_loss /= len(val_loader)
print(f'Validation MSE Loss: {val_loss:.4f}')

Validation MSE Loss: 0.8700


In [125]:
def predict_class(embedding, class_embeddings):
    embedding = embedding.unsqueeze(0)  # shape (1, embedding_dim)
    # Calculate Euclidean distance to each class embedding
    distances = torch.norm(class_embeddings - embedding, dim=1)  # shape (num_classes,)
    best_idx = torch.argmin(distances).item()  # smaller distance means closer
    return best_idx + 1

In [126]:
len(predictions)

786

In [127]:
val_df["embeddings"] = predictions

In [128]:
predicted_labels = []
for idx, row in val_df.iterrows():
    predicted_class = predict_class(
        torch.tensor(row["embeddings"]),
        torch.tensor(attributes)
    )
    predicted_labels.append(predicted_class)

In [129]:
val_df['predicted_label'] = predicted_labels

In [131]:
accuracy = accuracy_score(val_df['label'], val_df['predicted_label'])
print(accuracy)

0.015267175572519083


Run the model on test data

In [132]:
test_df = pd.read_csv("../data/test_images_path.csv")

In [133]:
test_dataset = ImageAttributeDataset(test_df, transform=test_tfms, attrs=attributes)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [134]:
model.eval()

ImageEmbeddingModel(
  (backbone): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=50176, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=65, bias=T

In [135]:
all_ids = test_df["id"].tolist()
all_preds = []

In [136]:
predictions = []
with torch.no_grad():
    for images, _ in test_loader:
        images, attrs = images.to(device), attrs.to(device)
        preds = model(images)
        predictions.extend(preds.tolist())

In [137]:
test_df = pd.DataFrame(
    columns=["id"], data={"id": all_ids}
)
test_df["embeddings"] = predictions
test_df["predicted_label"] = None
predicted_labels = []
for idx, row in test_df.iterrows():
    predicted_class = predict_class(
        torch.tensor(row["embeddings"]),
        torch.tensor(attributes)
    )
    predicted_labels.append(predicted_class)

In [138]:
test_df['label'] = predicted_labels

test_df[['id', 'label']].to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [110]:
len(train_df)

3140